<a href="https://colab.research.google.com/github/Crris07/Outamation_AI-Powered/blob/main/mini_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 53.6 MB/s eta 0:00:00


In [4]:
from google.colab import files
uploaded = files.upload()


Saving LenderFeesWorksheetNew.pdf to LenderFeesWorksheetNew.pdf


In [5]:
import fitz
import re

In [36]:
doc = fitz.open("LenderFeesWorksheetNew.pdf")
text = doc[0].get_text("text")
print(text)

Your actual rate, payment, and cost could be higher. Get an official Loan Estimate before choosing a loan.
Fee Details and Summary
Applicants:
Application No:
Date Prepared:
Loan Program:
Prepared By:
THIS IS NOT A GOOD FAITH ESTIMATE (GFE). This "Fees Worksheet" is provided for informational purposes ONLY, to assist
you in determining an estimate of cash that may be required to close and an estimate of your proposed monthly mortgage 
payment. Actual charges may be more or less, and your transaction may not involve a fee for every item listed.
Total Loan Amount:  
Interest Rate:
Term/Due In:
Fee
Paid To
Paid By (Fee Split**)
Amount
PFC / F / POC
TOTAL ESTIMATED FUNDS NEEDED TO CLOSE:
TOTAL ESTIMATED MONTHLY PAYMENT:
Total Estimated Funds
Total Monthly Payment
Purchase Price (+)
Alterations (+)
Land (+)
Refi (incl. debts to be paid off) (+)
Est. Prepaid Items/Reserves (+)
Est. Closing Costs (+)
Loan Amount (-)
Principal & Interest
Other Financing (P & I)
Hazard Insurance
Real Estate Tax

In [44]:
applicant_match = re.search(r"[A-Z][a-z]+\s[A-Z]\.\s[A-Z][a-z]+\s/\s[A-Z][a-z]+\s[A-Z]\.\s[A-Z][a-z]+", text)

if applicant_match:
    applicants = applicant_match.group()
    print(f"Applicants: {applicants}")
else:
    print("Applicants not found.")



rects = doc[0].search_for(applicants)
if rects:
    for i, rects in enumerate(rects):
        print(f"Found '{applicants}' at bounding box {rects}")
else:
     print(f"Bounding box for '{applicants}' not found.")

target_word = applicants


Applicants not found.
Found 'John Q. Smith / Mary A. Smith' at bounding box Rect(100.80000305175781, 78.44451904296875, 215.4901123046875, 87.37493133544922)


In [38]:
loan_match = re.search(r"\$\s+\d{3},\d{3}", text)
if loan_match:
    total_loan_amount = loan_match.group()
    print(f"Total Loan Amount: {total_loan_amount}")


    rects = doc[0].search_for(total_loan_amount)
    if rects:
        for i, rect in enumerate(rects):
            print(f"Found '{total_loan_amount}' at bounding box {rect}")
    else:
        print(f"Bounding box for '{total_loan_amount}' not found.")
else:
    print("Loan amount not found.")

Total Loan Amount: $ 380,000
Found '$ 380,000' at bounding box Rect(116.63999938964844, 161.96453857421875, 152.28775024414062, 170.89495849609375)


In [42]:
interest_rate = re.search(r"\d\.\d{3}\s%", text)

if interest_rate:
    interest = interest_rate.group()
    print(f"Interest rate: {interest}")

    rects = doc[0].search_for(interest)
    if rects:
        for i, rects in enumerate(rects):
            print(f"Found '{interest}' at bounding box {rects}")
    else:
        print(f"Bounding box for '{interest}' not found.")
else:
    print("Interest rate not found")




Interest rate: 4.250 %
Found '4.250 %' at bounding box Rect(298.79998779296875, 161.96453857421875, 328.2326354980469, 170.89495849609375)


In [63]:
# Applicants
applicants_phrase = "Applicants:"
rects_applicants = doc[0].search_for(applicants_phrase)
if rects_applicants:
    for i, rect in enumerate(rects_applicants):
        print(f"Found '{applicants_phrase}' at bounding box {rect}")
else:
    print(f"Bounding box for '{applicants_phrase}' not found.")

# Total Loan Amount
total_loan_amount_phrase = "Total Loan Amount:"
rects_loan_amount = doc[0].search_for(total_loan_amount_phrase)
if rects_loan_amount:
    for i, rect in enumerate(rects_loan_amount):
        print(f"Found '{total_loan_amount_phrase}' at bounding box {rect}")
else:
    print(f"Bounding box for '{total_loan_amount_phrase}' not found.")

# Interest Rate
interest_rate_phrase = "Interest Rate:"
rects_interest_rate = doc[0].search_for(interest_rate_phrase)
if rects_interest_rate:
    for i, rect in enumerate(rects_interest_rate):
        print(f"Found '{interest_rate_phrase}' at bounding box {rect}")
else:
    print(f"Bounding box for '{interest_rate_phrase}' not found.")

Found 'Applicants:' at bounding box Rect(39.599998474121094, 78.64180755615234, 77.63729858398438, 87.32872009277344)
Found 'Total Loan Amount:' at bounding box Rect(39.599998474121094, 161.84454345703125, 109.91036987304688, 170.77496337890625)
Found 'Interest Rate:' at bounding box Rect(237.47999572753906, 161.84454345703125, 285.7969665527344, 170.77496337890625)


In [74]:
import fitz
import re

def extract_tables_with_bboxes(doc):
    all_tables_data = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        tables = page.find_tables()

        for i, table in enumerate(tables.tables):
            table_data = []
            cell_bboxes = []

            for row_num in range(table.row_count):
                row_data = []
                row_bboxes = []

                for col_num in range(table.col_count):
                    cell_idx = row_num * table.col_count + col_num

                    if cell_idx < len(table.cells) and table.cells[cell_idx] is not None:
                        cell_bbox = table.cells[cell_idx]  # (x0, y0, x1, y1)
                        cell_text = page.get_text("text", clip=cell_bbox).strip()

                        row_data.append(cell_text)
                        row_bboxes.append(cell_bbox)
                    else:
                        row_data.append("")
                        row_bboxes.append(None)

                table_data.append(row_data)
                cell_bboxes.append(row_bboxes)

            all_tables_data.append({
                'page': page_num,
                'table_index': i,
                'bbox': table.bbox,
                'data': table_data,
                'cell_bboxes': cell_bboxes
            })
    return all_tables_data

doc = fitz.open("LenderFeesWorksheetNew.pdf")

extracted_tables = extract_tables_with_bboxes(doc)
doc.close()

print(f"Number of tables found: {len(extracted_tables)}")
if extracted_tables:
    for row in extracted_tables[0]['data'][:5]:
        print(row)

Number of tables found: 1
['Underwriting Fee\nXYZ Lender\nBorrower\n$\n550.00', 'Wire Transfer Fee\nXYZ Lender\nBorrower\n$\n75.00']
['Administration Fee\nXYZ Lender\nBorrower\n$\n445.00', '']
['', '']
